# Data Preparation and Feature Engineering

This notebook documents the **Data Preparation** phase of the CRISP-DM methodology for the *Mental Health in Technology* project.

The purpose of this stage is to transform the raw OSMI Mental Health in Tech 2016 survey data into an analytical representation that is suitable for subsequent dimensionality-reduction and unsupervised-clustering analysis.

The preceding notebook established that the raw dataset contains 1,433 observations and 63 variables with a mixture of numerical, binary, nominal, ordinal-like, and free-text responses. It also identified substantial and non-uniform missingness across the questionnaire.

This notebook therefore focuses on determining how the raw variables should be treated rather than applying a single transformation indiscriminately across all columns.

The preparation process will address:

- data-quality decisions identified during Data Understanding;
- missing-value treatment;
- variable-type classification;
- treatment of free-text and high-cardinality responses;
- categorical and binary representation;
- ordinal representation where a genuine ordering exists;
- feature engineering based on defensible information structure;
- removal of unsuitable or redundant variables;
- preparation of the analytical feature matrix.

No clustering model is fitted in this notebook.

The objective is to produce a transparent and reproducible feature representation that can be evaluated in the subsequent dimensionality-reduction and clustering stages.

## 1. Preparation Objectives

### Question

How should the raw survey variables be transformed into a defensible feature representation for unsupervised machine learning?

### Method

The preparation process follows the findings from the Data Understanding phase. Variables are first examined according to their statistical and semantic characteristics before transformations are selected.

The preparation decisions consider:

1. Whether a variable contains information relevant to the analytical objective.
2. Whether its response categories are nominal, binary, or genuinely ordered.
3. Whether missing values represent ordinary non-response or potentially meaningful conditional structure.
4. Whether a variable is free text or high-cardinality text that would require a separate text representation.
5. Whether a variable is redundant, unsuitable, or likely to introduce noise into clustering.
6. Whether the resulting representation remains interpretable.

### Why

Unsupervised clustering is sensitive to the representation of observations. Encoding decisions, scaling, missing-value treatment, and unnecessary variables can substantially influence distances and therefore cluster structure.

The preparation process must therefore be driven by the information contained in the survey rather than by a generic preprocessing recipe.

In [1]:
# Import necessary libraries
import pandas as pd

from pathlib import Path
from mental_health_ml.data import load_dataset

In [4]:
# Define the location of the immutable raw survey dataset.
RAW_DATA_PATH = Path(
    "../data/raw/mental-heath-in-tech-2016_20161114.csv.zip"
)

# Load the raw survey data without modifying the source observations.
df = load_dataset(RAW_DATA_PATH)

print(
    f"Raw dataset shape: {df.shape[0]:,} observations "
    f"x {df.shape[1]} variables."
)

Raw dataset shape: 1,433 observations x 63 variables.


## 2. Raw Data Integrity Check

### Question

Does the dataset entering Data Preparation retain the structure established during Data Understanding?

### Method

The observation count and variable count are checked against the raw-data findings established in Notebook 01.

### Why

The raw dataset should remain unchanged. Preparation should create new analytical representations rather than silently modifying the source data.

This provides a clear boundary between the original survey and all subsequent transformations.

In [5]:
if df.shape != (1433, 63):
    raise ValueError(
        f"Unexpected raw dataset shape: {df.shape}. "
        f"Expected (1433, 63)."
    )

print(
    "Raw-data integrity check passed: "
    f"{df.shape[0]:,} observations x {df.shape[1]:,} variables."
)

Raw-data integrity check passed: 1,433 observations x 63 variables.


## 3. Variable-Type Assessment

### Question

What types of information are represented by the 63 raw survey variables?

### Method

The variables are initially examined using their pandas data types and observed cardinality. This provides a starting point for distinguishing numerical variables, structured categorical responses, binary variables, ordinal-like responses, and high-cardinality free-text fields.

The initial data type does not by itself determine the final analytical representation. For example, several survey questions are stored as text but represent structured categorical responses rather than free-form language.

### Why

Different variable types require different representations.

Treating every text column as free text or applying one-hot encoding indiscriminately would ignore the semantic structure of the questionnaire and could create an unnecessarily large feature space.

In [7]:
# Create a compact structural summary for every raw variable.
variable_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non-nul": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=False),
    }
).sort_values(
    by="unique",
    ascending=False,
)

variable_summary.head(15)

,dtype,non-nul,missing,missing_pct,unique
Why or why not?,str,1095,338,23.586881,1086
Why or why not?.1,str,1126,307,21.423587,1081
Which of the following best describes your work position?,str,1433,0,0.000000,264
"If yes, what condition(s) have you been diagnosed with?",str,568,865,60.362875,129
"If so, what condition(s) were you diagnosed with?",str,711,722,50.383810,117
"If maybe, what condition(s) do you believe you have?",str,322,1111,77.529658,100
What is your gender?,str,1430,3,0.209351,71
What country do you live in?,str,1433,0,0.000000,53
What country do you work in?,str,1433,0,0.000000,53
What is your age?,int64,1433,0,0.000000,53


In [15]:
# Identify variables with very high observed cardinality
high_cardinality = variable_summary.loc[
    variable_summary["unique"] > 100
].copy()

high_cardinality[
    ["dtype", "missing", "missing_pct", "unique"]
]

,dtype,missing,missing_pct,unique
Why or why not?,str,338,23.586881,1086
Why or why not?.1,str,307,21.423587,1081
Which of the following best describes your work position?,str,0,0.000000,264
"If yes, what condition(s) have you been diagnosed with?",str,865,60.362875,129
"If so, what condition(s) were you diagnosed with?",str,722,50.383810,117


### Interpretation

The high-cardinality variables include free-text responses as well as demographic and employment variables.

The two variables beginning with `Why or why not?` contain more than 1,000 distinct responses. Their response structure is qualitatively different from the structured survey questions and should not be treated as ordinary categorical variables.

### Decision

The raw free-text `Why or why not?` variables will not be directly one-hot encoded into the primary clustering representation.

The primary analysis will focus on the structured survey variables whose response categories can be represented transparently using binary, nominal, or genuinely ordinal encodings.

This avoids generating a large sparse representation from open-ended responses without a sufficiently justified text-analysis objective.

In [12]:
# Identify variables that pandas currently represents numerically.
numeric_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

print(f"Numerical variables: {len(numeric_columns)}")

for column in numeric_columns:
    print(f"- {column}")

Numerical variables: 7
- Are you self-employed?
- Is your employer primarily a tech company/organization?
- Is your primary role within your company related to tech/IT?
- Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?
- Do you have previous employers?
- Have you ever sought treatment for a mental health issue from a mental health professional?
- What is your age?


In [13]:
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Are you self-employed?,1433.0,0.200279,0.400349,0.0,0.0,0.0,0.0,1.0
Is your employer primarily a tech company/organization?,1146.0,0.770506,0.420691,0.0,1.0,1.0,1.0,1.0
Is your primary role within your company related to tech/IT?,263.0,0.942966,0.232350,0.0,1.0,1.0,1.0,1.0
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,287.0,0.644599,0.479471,0.0,0.0,1.0,1.0,1.0
Do you have previous employers?,1433.0,0.882066,0.322643,0.0,1.0,1.0,1.0,1.0
Have you ever sought treatment for a mental health issue from a mental health professional?,1433.0,0.585485,0.492810,0.0,0.0,1.0,1.0,1.0
What is your age?,1433.0,34.286113,11.290931,3.0,28.0,33.0,39.0,323.0


## 4. Numerical Variable Validation

### Question

Do the numerical variables contain values that require correction before they are used analytically?

### Method

The observed ranges and summary statistics of numerical variables are examined before any transformation is applied.

### Why

Numerical values should not be assumed to be valid simply because pandas successfully parsed them as numbers. Age, for example, is a meaningful numerical variable only within a plausible range.

No numerical observation will be removed solely because it is unusual. Potential anomalies must first be distinguished from genuine but uncommon responses.

### Decision

Potentially invalid numerical observations will be investigated before any correction or exclusion is performed.

In [16]:
# Inspect the observed range of each numerical variable.
numeric_summary = pd.DataFrame(
    {
        "minimum": df[numeric_columns].min(),
        "maximum": df[numeric_columns].max(),
        "median": df[numeric_columns].median(),
        "missing": df[numeric_columns].isna().sum(),
    }
)

numeric_summary

,minimum,maximum,median,missing
Are you self-employed?,0.0,1.0,0.0,0
Is your employer primarily a tech company/organization?,0.0,1.0,1.0,287
Is your primary role within your company related to tech/IT?,0.0,1.0,1.0,1170
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,0.0,1.0,1.0,1146
Do you have previous employers?,0.0,1.0,1.0,0
Have you ever sought treatment for a mental health issue from a mental health professional?,0.0,1.0,1.0,0
What is your age?,3.0,323.0,33.0,0


In [17]:
age_column = "What is your age?"

age_summary = df[age_column].describe()
age_summary

count    1433.000000
mean       34.286113
std        11.290931
min         3.000000
25%        28.000000
50%        33.000000
75%        39.000000
max       323.000000
Name: What is your age?, dtype: float64

In [18]:
# Display the lowest and highest observed ages for direct inspection.
df[age_column].sort_values().head(20), df[age_column].sort_values().tail(10)

(656      3
 808     15
 93      17
 76      19
 736     19
 754     19
 559     19
 989     20
 528     20
 282     20
 1268    20
 771     20
 1292    20
 1301    21
 1322    21
 588     21
 655     21
 806     21
 143     21
 306     21
 Name: What is your age?, dtype: int64,
 900      63
 308      63
 1224     63
 745      63
 885      65
 701      66
 1221     70
 1117     74
 372      99
 564     323
 Name: What is your age?, dtype: int64)

### Interpretation

The observed age values are inspected before any preparation decision is made.

Extreme values must be distinguished between:

- plausible but uncommon respondent ages;
- values that are clearly inconsistent with the meaning of the question; and
- missing observations.

### Decision

Only observations that can be justified as invalid representations of age will be treated as data-quality issues. Unusual but plausible ages will not be removed merely because they are statistically uncommon.

Any correction or exclusion will be documented explicitly in the preparation logic.

## 5. Missing-Data Treatment

### Question

How should missing responses be handled without confusing non-response with meaningful conditional survey structure?

### Method

Missingness is examined by variable and compared with the questionnaire structure identified during Data Understanding.

Variables are not automatically imputed simply because they contain missing values.

### Why

Several survey questions are conditional. For example, questions concerning employer coverage, treatment, diagnosis, or disclosure may not apply to every respondent.

Consequently, a missing response can arise because:

- the respondent did not answer the question;
- the question was conditionally presented;
- the question was not applicable;
- the respondent reached a different branch of the questionnaire.

Treating every missing value as the same phenomenon could introduce artificial information into the feature space.

### Decision

Missing-value treatment will therefore be determined by variable meaning and response structure rather than by a single global imputation rule.

In [19]:
missing_summary = (
    pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": df.isna().mean().mul(100),
        }
    )
    .sort_values(
        by="missing_pct",
        ascending=False,
    )
)

missing_summary.head(20)

,missing_count,missing_pct
"If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?",1289,89.951151
"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",1229,85.764131
Is your primary role within your company related to tech/IT?,1170,81.646895
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,1146,79.972087
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?",1146,79.972087
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?",1146,79.972087
"If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",1146,79.972087
Do you know local or online resources to seek help for a mental health disorder?,1146,79.972087
Do you believe your productivity is ever affected by a mental health issue?,1146,79.972087
"If maybe, what condition(s) do you believe you have?",1111,77.529658


In [20]:
# Count the number of missing values in each respondent record.
row_missing_counts = df.isna().sum(axis=1)

row_missing_counts.describe()

count    1433.000000
mean       15.324494
std         5.210386
min         9.000000
25%        12.000000
50%        14.000000
75%        18.000000
max        37.000000
dtype: float64

### Interpretation

The distribution of missing values across respondents helps determine whether missingness is concentrated in particular questionnaire sections or is broadly distributed across the survey.

A repeated missingness pattern across many variables may indicate questionnaire branching or conditional applicability rather than random item non-response.

### Decision

Missingness patterns will be incorporated into the preparation decisions for individual variables. The analysis will not assume that all missing values have identical statistical meaning.

A blanket mean or mode imputation strategy is not applied at this stage.

For categorical survey variables, replacing missing responses with the most frequent category would manufacture responses that were not provided by participants and could distort the distribution of relatively uncommon categories.

For numerical variables, imputation will only be considered where the variable is retained in the analytical representation and where the method is justified by the observed data structure.

Where missingness corresponds to a meaningful non-applicability state, that distinction will be preserved rather than converted into an arbitrary value.

In [21]:
# Count distinct non-missing responses for each variable.
non_missing_unique_counts = df.nunique(dropna=True)

binary_candidates = non_missing_unique_counts[
    non_missing_unique_counts == 2
].index.tolist()

print(
    "Variables with exactly two non-missing response categories: "
    f"{len(binary_candidates)}"
)

for column in binary_candidates:
    print(f"- {column}")

Variables with exactly two non-missing response categories: 8
- Are you self-employed?
- Is your employer primarily a tech company/organization?
- Is your primary role within your company related to tech/IT?
- Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?
- Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?
- Do you have previous employers?
- Have you been diagnosed with a mental health condition by a medical professional?
- Have you ever sought treatment for a mental health issue from a mental health professional?


## 6. Ordinal Representation

### Question

Which categorical variables contain a genuine ordered response structure?

### Method

Variables are reviewed for response categories where the survey wording establishes a meaningful progression from lower to higher levels of the measured concept.

### Why

Ordinal variables contain information that ordinary nominal one-hot encoding does not explicitly represent. Where a genuine ordering exists, an ordinal representation can preserve that information more efficiently.

However, arbitrary numerical ordering must not be imposed on nominal categories.

### Decision

Only variables with a defensible semantic ordering will be considered for ordinal encoding. Nominal responses such as `Yes`, `No`, and `I don't know` will not be assigned arbitrary numerical ranks.

In [22]:
text_columns = [
    column
    for column in df.columns
    if column.startswith("Why or why not?")
]

text_columns

['Why or why not?', 'Why or why not?.1']

### Multi-response Work Position

The work-position variable contains observations with multiple role descriptions separated by a delimiter.

Treating each complete combination as a separate categorical level would create a high-cardinality feature and would treat equivalent role combinations as unrelated categories.

The variable therefore requires dedicated feature engineering before it can be considered for the analytical representation.

The response components will be examined to determine whether a small number of meaningful role indicators can be derived without creating an excessive number of artificial features.

In [23]:
position_column = "Which of the following best describes your work position?"

position_values = (
    df[position_column]
    .dropna()
    .astype(str)
    .str.split("|")
    .explode()
    .str.strip()
)

position_frequency = position_values.value_counts()

position_frequency

Which of the following best describes your work position?
Back-end Developer         737
Front-end Developer        502
DevOps/SysAdmin            282
Supervisor/Team Lead       277
Other                      187
Support                    168
One-person shop            161
Designer                   135
Executive Leadership       101
Dev Evangelist/Advocate     99
Sales                       31
HR                          12
Name: count, dtype: int64

## 7. Feature Engineering Principles

Feature engineering is restricted to transformations that preserve meaningful information contained in the survey.

The following principles are applied:

- Do not create features merely to increase dimensionality.
- Preserve meaningful distinctions between survey responses.
- Use binary indicators where a response represents the presence or absence of a characteristic.
- Use ordinal representations only where a genuine ordering exists.
- Represent multi-response variables through meaningful component indicators where justified.
- Avoid arbitrary numerical coding of nominal categories.
- Avoid incorporating raw free-text responses without a justified text-analysis objective.
- Avoid duplicate representations of the same information.
- Keep the resulting feature space interpretable enough to support later cluster profiling.

Every engineered feature must have a clear relationship to the underlying survey question.